# 05 · Gold — Fato Veículos | Acidentes ANTT

| | |
|---|---|
| **Origem** | Delta Table `silver_acidentes` + dims `gold_dim_*` |
| **Destino** | Delta Table `gold_fato_veiculo_acidente` |
| **Execução no Pipeline** | Etapa 2 (paralelo) — requer `03_nb_gold_dims` completo |
| **Idempotente** | Sim — `overwrite` recria a tabela |

**Unpivot:** 10 colunas de veículo (wide) → `(tipo_veiculo, quantidade)` (long).
Somente linhas com `quantidade > 0` são gravadas.

**Schema:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `id_acidente` | long | Dimensão degenerada — sem relacionamento no Power BI |
| `id_data` | int | FK → `gold_dim_data` |
| `id_concessionaria` | int | FK → `gold_dim_concessionaria` |
| `id_rodovia` | int | FK → `gold_dim_rodovia` |
| `id_veiculo` | int | FK → `gold_dim_veiculo` |
| `quantidade` | int | Número de veículos desse tipo no acidente |

## 1 · Imports

In [ ]:
from typing import List

from pyspark.sql import DataFrame, functions as F

## 2 · Parâmetros

> Célula marcada como **Parameters** — valores podem ser sobrescritos por um Data Pipeline.

In [ ]:
NOTEBOOK_NAME: str = "gold_fato_veiculos"
TABELA_SILVER: str = "silver_acidentes"
PREFIXO_GOLD:  str = "gold_"
MODO_ESCRITA:  str = "overwrite"
LOG_LEVEL:     str = "INFO"

## 3 · Configuração Compartilhada

In [ ]:
%run 00_nb_config

In [ ]:
# ── Otimizações Spark ─────────────────────────────────────────────────────────
# AQE: replaneja joins e reparticiona resultados em tempo de execucao (Spark 3.3+)
spark.conf.set("spark.sql.adaptive.enabled",                    "true")
# coalescePartitions: AQE consolida particoes pequenas dinamicamente apos shuffles
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Dimensoes <= 100 MB viram broadcast join automaticamente (evita sort-merge join)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",          str(100 * 1024 * 1024))
log.info("Spark: AQE=on | coalescePartitions=on | broadcast_threshold=100MB")

## 4 · Leitura Silver e Dimensões

In [ ]:
COLS_VEICULOS: List[str] = [
    "automovel", "bicicleta", "caminhao", "moto", "onibus",
    "outros", "tracao_animal", "transporte_de_cargas_especiais",
    "trator_maquinas", "utilitarios",
]

# Silver com surrogate key deterministico (mesmo algoritmo nos 3 notebooks de fato)
df = (
    spark.table(TABELA_SILVER)
    .withColumn(
        "id_acidente",
        F.xxhash64(
            F.col("data"),
            F.col("concessionaria"),
            F.col("km"),
            F.col("hora"),
            F.col("tipo_de_acidente"),
        ),
    )
    .cache()
)
log.info("Silver lida e cacheada: %d registros", df.count())

# Dims compartilhadas ja persistidas por 03_nb_gold_dims
dim_data           = spark.table(f"{PREFIXO_GOLD}dim_data")
dim_concessionaria = spark.table(f"{PREFIXO_GOLD}dim_concessionaria")
dim_rodovia        = spark.table(f"{PREFIXO_GOLD}dim_rodovia")
dim_veiculo        = spark.table(f"{PREFIXO_GOLD}dim_veiculo")
log.info("Dimensões carregadas.")

## 5 · Unpivot e Construção do Fato

In [ ]:
# F.broadcast(): hint explicito nas dims compartilhadas — evita sort-merge join
df_fk = (
    df
    .join(F.broadcast(dim_data.select("data", "id_data")),
          on="data", how="left")
    .join(F.broadcast(dim_concessionaria.select("concessionaria", "id_concessionaria")),
          on="concessionaria", how="left")
    .join(F.broadcast(dim_rodovia.select("rodovia", "uf", "id_rodovia")),
          on=["rodovia", "uf"], how="left")
    # ano: coluna de particao (pruning em queries filtradas por ano no Power BI)
    .withColumn("ano", (F.col("id_data") / 10000).cast("int"))
    .select("id_acidente", "id_data", "ano", "id_concessionaria", "id_rodovia",
            *COLS_VEICULOS)
)

# Unpivot: wide (10 colunas) → long (tipo_veiculo, quantidade)
n = len(COLS_VEICULOS)
stack_expr = ", ".join(f"'{c}', {c}" for c in COLS_VEICULOS)

fato_veiculo_acidente = (
    df_fk
    .selectExpr(
        "id_acidente", "id_data", "ano", "id_concessionaria", "id_rodovia",
        f"stack({n}, {stack_expr}) as (tipo_veiculo, quantidade)",
    )
    .filter(F.col("quantidade") > 0)
    # broadcast na dim_veiculo (10 linhas — candidato ideal)
    .join(F.broadcast(dim_veiculo.select("tipo_veiculo", "id_veiculo")),
          on="tipo_veiculo", how="left")
    .select("id_acidente", "id_data", "ano", "id_concessionaria",
            "id_rodovia", "id_veiculo", "quantidade")
)

df.unpersist()
log.info("fato_veiculo_acidente construido: %d linhas", fato_veiculo_acidente.count())

## 6 · Persistência

In [ ]:
nome = f"{PREFIXO_GOLD}fato_veiculo_acidente"
(
    fato_veiculo_acidente.write
    .format("delta")
    .mode(MODO_ESCRITA)
    .option("overwriteSchema", "true")
    .partitionBy("ano")          # pruning de particao em queries filtradas por ano
    .saveAsTable(nome)
)
total = spark.sql(f"SELECT COUNT(*) FROM {nome}").collect()[0][0]
log.info("%s salva: %d registros", nome, total)

# OPTIMIZE + ZORDER pelas colunas mais usadas em filtros do Power BI
spark.sql(f"OPTIMIZE {nome} ZORDER BY (id_data, id_concessionaria, id_veiculo)")
log.info("OPTIMIZE ZORDER aplicado em %s", nome)

## 7 · Relatório

In [ ]:
log.info("=== Top 10 tipos de veiculo mais envolvidos ===")
spark.sql(f"""
    SELECT v.tipo_veiculo, SUM(f.quantidade) AS total
    FROM {PREFIXO_GOLD}fato_veiculo_acidente f
    JOIN {PREFIXO_GOLD}dim_veiculo v ON f.id_veiculo = v.id_veiculo
    GROUP BY v.tipo_veiculo
    ORDER BY total DESC
""").show(truncate=False)

notebookutils.notebook.exit(f"OK: {PREFIXO_GOLD}fato_veiculo_acidente criada com {total} registros.")